
# Narrative Similarity — Track B (Single-Story Embeddings) 📘

This notebook builds **Track B** embeddings for the Narrative Similarity task using **only each story itself at inference** (Track-B compliant).  
It supports:
- **Aspect-aware text construction** from a single story (Theme / Action / Outcome).
- **(Optional) Fine-tuning** a Sentence-Transformers model on training triples (anchor, A, B).
- **Export** to `track_b.jsonl` and `track_b.npy` in dataset order.
- **Dev evaluation**: accuracy on dev triples.

> **Inputs expected** (adjust paths if needed):
> - `data/train_triples.jsonl` (optional, for training)  
> - `data/dev_triples.jsonl` (optional, for evaluation)  
> - `data/items.jsonl` (required for Track B export; one story per line with key `text`)


In [ ]:

# %%capture
# If needed, uncomment to install deps
# !pip install -U sentence-transformers tqdm numpy pandas scikit-learn


In [ ]:

import os, json, re, math, random, string, time, sys
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

USE_SENTENCE_TRANSFORMERS = True        # Set False to use a TF-IDF/Hashing fallback only
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # lightweight and solid
SEED = 42

DATA_DIR = Path("data")
TRAIN_TRIPLES = DATA_DIR / "train_triples.jsonl"  # optional
DEV_TRIPLES   = DATA_DIR / "dev_triples.jsonl"    # optional
ITEMS_FILE    = DATA_DIR / "items.jsonl"          # required for final export

OUT_JSONL = Path("track_b.jsonl")
OUT_NPY   = Path("track_b.npy")

random.seed(SEED)
np.random.seed(SEED)



## Utilities — Aspect extraction (Theme / Action / Outcome)

Heuristic, **LLM-free** extractors that rely on sentence boundaries and keywords (keeps inference strictly single-story).
You can later swap these with a better extractor (SRL/LLM) as long as inference uses only the single story.


In [ ]:

SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')

def split_sentences(text: str) -> List[str]:
    text = (text or "").strip()
    if not text:
        return []
    sents = SENT_SPLIT_RE.split(text)
    return [s.strip() for s in sents if s.strip()]

def pick_theme(text: str, max_sents: int = 3) -> str:
    """Pick first ~2-3 sentences as 'Theme' proxy (problem/stakes often introduced early)."""
    sents = split_sentences(text)
    return " ".join(sents[:max_sents]) if sents else text

def pick_outcome(text: str, max_sents: int = 2) -> str:
    """Pick the final 1-2 sentences as 'Outcome' proxy (resolution usually near the end)."""
    sents = split_sentences(text)
    return " ".join(sents[-max_sents:]) if sents else text

ACTION_VERB_CUES = {
    "go","goes","went","travel","travels","traveled","reach","reaches","reached",
    "fight","fights","fought","escape","escapes","escaped","search","searches","searched",
    "find","finds","found","discover","discovers","discovered","return","returns","returned",
    "kill","kills","killed","die","dies","died","plan","plans","planned","attack","attacks","attacked",
    "try","tries","tried","start","starts","started","begin","begins","began","decide","decides","decided",
    "save","saves","saved","help","helps","helped","learn","learns","learned","investigate","investigates","investigated",
    "betray","betrays","betrayed","lose","loses","lost","win","wins","won"
}

def extract_action_chain(text: str, max_items: int = 6) -> str:
    """Very light heuristic 'action chain': choose sentences containing action-like verbs, preserve order."""
    sents = split_sentences(text)
    picked = []
    for s in sents:
        toks = re.findall(r"[A-Za-z']+", s.lower())
        if any(t in ACTION_VERB_CUES or t.endswith("ed") or t.endswith("ing") for t in toks):
            picked.append(s)
        if len(picked) >= max_items:
            break
    if not picked and sents:
        picked = sents[:min(3, len(sents))]
    # Compress a bit
    return " | ".join(picked)

def build_aspect_text(story_text: str) -> str:
    theme = pick_theme(story_text, max_sents=3)
    action = extract_action_chain(story_text, max_items=6)
    outcome = pick_outcome(story_text, max_sents=2)
    return f"THEME: {theme}\nACTION: {action}\nOUTCOME: {outcome}"



## Embedding backends

Default: Sentence-Transformers (`all-MiniLM-L6-v2`).  
Fallback: Hashing-based TF-IDF to 1024 dims (no training required).


In [ ]:

MODEL = None
TOKENIZER = None

def load_st_model():
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(BASE_MODEL)
    return model

def embed_st(texts: List[str]) -> np.ndarray:
    assert MODEL is not None, "MODEL is None; load it first."
    return np.asarray(MODEL.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True))

# Hashing fallback (deterministic)
import hashlib
from collections import Counter

HF_DIM = 1024
def hash_index(token, D=HF_DIM): return int(hashlib.md5(token.encode()).hexdigest(), 16) % D
def hash_sign(token): return 1 if (int(hashlib.sha1(token.encode()).hexdigest(), 16) % 2)==0 else -1
WORD_RE = re.compile(r"[A-Za-z']+")

def tokenize(text: str) -> List[str]:
    return WORD_RE.findall(text.lower())

def ngrams(tokens, n):
    return ["_".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def embed_hashing(texts: List[str]) -> np.ndarray:
    # Build IDF from the batch (per-batch proxy)
    docs_tokens = []
    df = Counter()
    for s in texts:
        toks = tokenize(s)
        feats = toks + ngrams(toks, 2)
        docs_tokens.append(feats)
        df.update(set(feats))
    N = len(texts)
    idf = {t: math.log((1+N)/(1+df[t])) + 1.0 for t in df}
    X = np.zeros((N, HF_DIM), dtype=np.float32)
    for i, feats in enumerate(docs_tokens):
        tf = Counter(feats)
        v = np.zeros(HF_DIM, dtype=np.float32)
        for t, c in tf.items():
            w = c * idf.get(t, 1.0)
            idx = hash_index(t)
            v[idx] += hash_sign(t) * w
        nrm = np.linalg.norm(v) + 1e-12
        X[i] = v / nrm
    return X



## (Optional) Fine-tuning on training triples

Uses `MultipleNegativesRankingLoss`: for each (anchor, chosen) pair, other items in-batch serve as negatives.  
We train on **aspect-serialized text** (`THEME\nACTION\nOUTCOME`) so the model learns to align the three aspects.


In [ ]:

DO_TRAIN = False   # <- flip to True if you have TRAIN_TRIPLES
EPOCHS   = 3
BATCH    = 64
LR       = 2e-5
WARMUP   = 0.1

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    items = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: 
                continue
            items.append(json.loads(line))
    return items

def build_training_examples(triples: List[Dict[str, Any]]) -> List[Tuple[str, str]]:
    """Return list of (anchor_aspect_text, chosen_aspect_text). Expects keys: anchor, A, B, text_a_is_closer."""
    pairs = []
    for t in triples:
        anchor = t.get('anchor') or t.get('Anchor') or t.get('anchor_text') or t.get('anchorStory') or ""
        A = t.get('A') or t.get('a') or t.get('candA') or ""
        B = t.get('B') or t.get('b') or t.get('candB') or ""
        label = t.get('text_a_is_closer')
        if label is None:
            # fallback: maybe 'label' == 'A' or 'B'
            lab = t.get('label')
            if isinstance(lab, str):
                label = (lab.strip().upper() == 'A')
        if anchor is None or A is None or B is None:
            continue
        chosen = A if label else B
        pairs.append((build_aspect_text(anchor), build_aspect_text(chosen)))
    return pairs

def fine_tune(model, train_pairs: List[Tuple[str, str]], epochs=EPOCHS, batch_size=BATCH, lr=LR, warmup=WARMUP):
    from sentence_transformers import losses, InputExample, datasets
    train_examples = [InputExample(texts=[a, b]) for a, b in train_pairs]
    train_dl = datasets.NoDuplicatesDataLoader(train_examples, batch_size=batch_size)
    train_loss = losses.MultipleNegativesRankingLoss(model)
    warmup_steps = int(len(train_dl) * epochs * warmup)
    model.fit(train_objectives=[(train_dl, train_loss)],
              epochs=epochs,
              warmup_steps=warmup_steps,
              scheduler='linear',
              optimizer_params={'lr': lr},
              show_progress_bar=True)
    return model

# Kick off (only if DO_TRAIN)
if USE_SENTENCE_TRANSFORMERS:
    MODEL = load_st_model()
    if DO_TRAIN and TRAIN_TRIPLES.exists():
        triples = read_jsonl(TRAIN_TRIPLES)
        train_pairs = build_training_examples(triples)
        print(f"Training on {len(train_pairs)} pairs...")
        MODEL = fine_tune(MODEL, train_pairs)
    else:
        print("Loaded base model without fine-tuning.")
else:
    print("Using hashing fallback; no training.")



## Dev evaluation (accuracy on triples)

For each triple, we compare cosine(anchor, A) vs cosine(anchor, B) using aspect-serialized embeddings.


In [ ]:

def st_embed_texts(texts: List[str]) -> np.ndarray:
    if USE_SENTENCE_TRANSFORMERS:
        return embed_st(texts)
    else:
        return embed_hashing(texts)

def accuracy_on_triples(triples_path: Path) -> float:
    triples = read_jsonl(triples_path)
    if not triples:
        print("No dev triples found.")
        return 0.0
    rights = 0
    for t in tqdm(triples, desc="Evaluating"):
        anchor = t.get('anchor') or t.get('Anchor') or t.get('anchor_text') or t.get('anchorStory') or ""
        A = t.get('A') or t.get('a') or t.get('candA') or ""
        B = t.get('B') or t.get('b') or t.get('candB') or ""
        label = t.get('text_a_is_closer')
        if label is None:
            lab = t.get('label')
            if isinstance(lab, str):
                label = (lab.strip().upper() == 'A')
        if not anchor or not A or not B or label is None:
            continue
        a_texts = [build_aspect_text(anchor), build_aspect_text(A), build_aspect_text(B)]
        emb = st_embed_texts(a_texts)
        anc, eA, eB = emb[0], emb[1], emb[2]
        simA = float(np.dot(anc, eA))
        simB = float(np.dot(anc, eB))
        pred_is_A = simA > simB
        rights += int(pred_is_A == bool(label))
    acc = rights / len(triples)
    print(f"Dev accuracy: {acc:.4f} on {len(triples)} items")
    return acc

if DEV_TRIPLES.exists():
    _ = accuracy_on_triples(DEV_TRIPLES)
else:
    print("DEV_TRIPLES not found; skipping eval.")



## Track B export

Reads `data/items.jsonl` (with a `text` field per line, in **exact** submission order) and writes:
- `track_b.jsonl` (one embedding per line as `{"embeddings": [...]}`)
- `track_b.npy` (float32 matrix)


In [ ]:

def export_track_b(items_path: Path, out_jsonl: Path, out_npy: Path):
    assert items_path.exists(), f"Missing items file: {items_path}"
    stories = []
    with items_path.open('r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): 
                continue
            obj = json.loads(line)
            # Expect a key named 'text'; adjust here if your key differs
            stories.append(obj.get('text') or obj.get('story') or obj.get('content') or "")
    aspect_texts = [build_aspect_text(s) for s in stories]
    X = st_embed_texts(aspect_texts).astype('float32')

    # JSONL
    with out_jsonl.open('w', encoding='utf-8') as f:
        for row in X.tolist():
            f.write(json.dumps({"embeddings": row}) + "\n")
    # NPY
    np.save(out_npy, X)
    print(f"Wrote {len(stories)} embeddings → {out_jsonl} and {out_npy}")
    return X

# Run export if items exist
if ITEMS_FILE.exists():
    _ = export_track_b(ITEMS_FILE, OUT_JSONL, OUT_NPY)
else:
    print("ITEMS_FILE not found; add your dataset to data/items.jsonl and re-run this cell.")



## (Optional) Smoke test with ad hoc texts

You can paste a few stories here and see nearest neighbors.


In [ ]:

TEST_STORIES = [
    "A woman loses a locket on the way to a wedding, searches, then a stranger returns it; the wedding proceeds.",
    "A squad is stranded on the moon after a battle and must trek back to extraction before oxygen runs out.",
    "A town plans to bulldoze a historic neighborhood; locals fight a land battle to stop the development."
]

if TEST_STORIES:
    asp = [build_aspect_text(t) for t in TEST_STORIES]
    E = st_embed_texts(asp)
    S = E @ E.T
    print("Cosine similarity matrix:\n", np.round(S, 3))
